In [ ]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'figure.dpi':150,'reso':'xx-hi',
    'font.size':8,'label.size':8,'tick.labelsize':8,
    'leftlabelsize':8,'toplabelsize':8,
    'leftlabel.weight':'normal','toplabel.weight':'normal',
    'tick.minor':False})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR = CONFIGS['filepaths']['splits']
PREDSDIR  = CONFIGS['filepaths']['predictions']
SPLIT     = 'valid'

# Predictors to show: listed in display order
PREDICTORS = [
    ('rh',         True,  r'$\widehat{RH}$ (normalized)'),
    ('thetae',     True,  r'$\widehat{\theta_e}$ (normalized)'),
    ('thetaestar', True,  r'$\widehat{\theta_e^*}$ (normalized)'),
    ('shf',        False, r'$\widehat{SHF}$ (normalized)'),
    ('lf',         False, r'Land Fraction (normalized)'),
]  # (varname, has_sig_dim, xlabel)

MODELS_8 = ['pod_bl','base_bl','base_all','nonparam_all','gauss_all','sr_rh','sr_thermo','sr_thermo_flux']
LABELS_8 = {
    'pod_bl':        'POD',
    'base_bl':       'Baseline NN ($B_L$)',
    'base_all':      'Baseline NN (all)',
    'nonparam_all':  'Nonparam. NN',
    'gauss_all':     'Param. NN',
    'sr_rh':         'SR-RH',
    'sr_thermo':     'SR-Thermo',
    'sr_thermo_flux':'SR-Thermo+Flux',
}
# Color + linestyle per model
STYLES = {
    'pod_bl':        ('blood',    '-',  1.8),
    'base_bl':       ('#5BA7DA', '--',  1.4),
    'base_all':      ('#5BA7DA',  '-',  1.8),
    'nonparam_all':  ('#F2C85E', '--',  1.4),
    'gauss_all':     ('#D42028',  '-',  1.8),
    'sr_rh':         ('#1B2C61',  ':',  1.4),
    'sr_thermo':     ('#1B2C61', '--',  1.4),
    'sr_thermo_flux':('#1B2C61',  '-',  1.8),
}
NBINS = 50

In [ ]:
# Load normalized split for predictor values
with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    normds = {v:ds[v].load() for v in ds.data_vars}

# Build flat predictor arrays (time*lat*lon,)
ref_dims = ('time','lat','lon')
refshape = tuple(normds['tp'].sizes[d] for d in ref_dims)
refsize  = int(np.prod(refshape))

pred_flat = {}  # varname -> 1D array
for varname,has_sig,_ in PREDICTORS:
    if varname not in normds:
        print(f'Warning: {varname} not in norm split, skipping')
        continue
    da   = normds[varname]
    if has_sig and 'sig' in da.dims:
        da = da.mean('sig')
    da = da.transpose(*[d for d in ref_dims if d in da.dims])
    arr = da.values.ravel()
    if arr.size == refsize:
        pred_flat[varname] = arr
    else:
        # broadcast local vars to match ref grid
        full = np.broadcast_to(da.values[...,np.newaxis] if 'lon' not in da.dims else da.values, refshape)
        pred_flat[varname] = full.ravel()

# Load observations
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    truetp = ds.tp.load()
true_flat = truetp.transpose(*ref_dims).values.ravel()

# Load predictions for each model
model_flat = {}  # name -> 1D array in mm/day
for name in MODELS_8:
    fpath = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    if not os.path.exists(fpath):
        print(f'Missing predictions: {name}')
        continue
    with xr.open_dataset(fpath) as ds:
        pred = ds.tp.load()
    if 'seed' in pred.dims:
        pred = pred.mean('seed')
    if 'complexity' in pred.dims:
        pred = pred.isel(complexity=0)
    _,pred = xr.align(truetp,pred,join='inner')
    model_flat[name] = pred.squeeze().transpose(*ref_dims).values.ravel()

valid_mask = np.isfinite(true_flat)
print(f'Valid samples: {valid_mask.sum():,}  |  Models loaded: {len(model_flat)}')

In [ ]:
def bin_conditional(x_flat,y_flat,nbins=50):
    '''Bin y by quantiles of x; return (bin_centers, bin_means, bin_counts).'''
    finite    = np.isfinite(x_flat) & np.isfinite(y_flat)
    x,y       = x_flat[finite],y_flat[finite]
    edges     = np.quantile(x,np.linspace(0,1,nbins+1))
    centers   = 0.5*(edges[:-1]+edges[1:])
    means,counts = np.full(nbins,np.nan),np.zeros(nbins,dtype=int)
    for i in range(nbins):
        sel = (x >= edges[i]) & (x <= edges[i+1])
        if sel.sum() >= 5:
            means[i]  = y[sel].mean()
            counts[i] = sel.sum()
    return centers,means,counts

avail_preds = [(vn,has_sig,xl) for vn,has_sig,xl in PREDICTORS if vn in pred_flat]
ncols       = len(avail_preds)
fig,axs     = pplt.subplots(ncols=ncols,refwidth=2.2,refheight=2.5,share=False)
axs.format(
    grid=True,gridminor=False,
    ylabel='Mean Precipitation (mm/day)',
    suptitle=f'Conditional Mean Precipitation by Predictor — {SPLIT} set')
axs = np.atleast_1d(axs).ravel()

for col,(varname,has_sig,xlabel) in enumerate(avail_preds):
    ax   = axs[col]
    xarr = pred_flat[varname]

    # Observed
    centers,obs_means,_ = bin_conditional(xarr,true_flat,NBINS)
    ax.plot(centers,obs_means,color='k',lw=2,ls='-',label='Observed',zorder=10)

    # Each model
    for name in MODELS_8:
        if name not in model_flat:
            continue
        color,ls,lw = STYLES[name]
        _,means,_   = bin_conditional(xarr,model_flat[name],NBINS)
        ax.plot(centers,means,color=color,ls=ls,lw=lw,
                label=LABELS_8.get(name,name),zorder=5)

    ax.format(xlabel=xlabel,xminorticks='none',yminorticks='none')

axs[0].legend(loc='ul',ncols=1,fontsize=7)
pplt.show()
# fig.save('../figs/fig_partial_dependence.jpg')